# Embeddings & evaluation

A focused companion to [`walkthrough.ipynb`](walkthrough.ipynb). The walkthrough runs the whole
pipeline end to end; this one zooms in on two ideas:

1. **What an embedding is** — text turned into a vector so similar meanings sit near each other.
2. **How to measure retrieval** — `hit@k` and `MRR` — and how those numbers tell you when a
   stronger (hosted) embedder is actually worth the cost.

It runs **key-free**: the default local `model2vec` embedder needs no API key, and nothing here
calls generation, so nothing is billed. (`config` requires an `ANTHROPIC_API_KEY` to merely be
*present* at import — a placeholder is fine; we never use it.)

Launch from the repo root: `uv run jupyter lab`.

In [ ]:
import os
from pathlib import Path

# Resolve the repo root so ./data/index and ./eval/gold work whether Jupyter was
# launched at the root or inside notebooks/.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
    os.chdir(ROOT)

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
# config.py requires ANTHROPIC_API_KEY at import; this notebook does only embeddings +
# retrieval + eval (no generation), so a placeholder is enough and nothing is billed.
os.environ.setdefault("ANTHROPIC_API_KEY", "placeholder-not-used-in-this-notebook")
print("repo root:", ROOT)

## 1. What an embedding is

An embedding turns text into a vector so that **similar meanings land near each other** — and the
*same* model embeds both the stored rules and your question, or the geometry wouldn't line up.
Below, embed **two pairs of near-paraphrases** — two ways of saying one checkers idea, two ways of
saying one cornhole idea — and look at how close they are (cosine similarity; 1.0 = same direction).

In [ ]:
import numpy as np
from rulebook.embeddings import get_embedder

emb = get_embedder()  # local model2vec by default — no key, no call to a hosted provider
phrases = [
    "winning a game of checkers",         # p0 ─┐ same idea,
    "how you win at checkers",            # p1 ─┘ said two ways
    "tossing a beanbag onto the cornhole board",  # p2 ─┐ same idea,
    "throwing a bag in cornhole",                 # p3 ─┘ said two ways
]
vecs = np.asarray(emb.embed(phrases, input_type="document"))
sims = vecs @ vecs.T  # unit-length vectors, so the dot product IS cosine similarity

print("      " + "   ".join(f"p{i}" for i in range(len(phrases))))
for i, row in enumerate(sims):
    print(f"p{i}  " + "  ".join(f"{v:5.2f}" for v in row))
print()
for i, p in enumerate(phrases):
    print(f"p{i}: {p}")

Each phrase is nearest its paraphrase partner — `p0↔p1` and `p2↔p3` are the biggest off-diagonal
numbers, and similarity drops across the two topics. That's the whole idea: *meaning*, not matching
words, decides nearness. (It's an imperfect, blunt signal here — this is a lightweight static model,
a limitation section 5 leans on.)

## 2. A tiny index to search

Retrieval needs an index: every rule chunked and embedded. Build the two bundled sample domains
if they aren't already — this is exactly what the quickstart's `build_index.py` does, key-free.

In [ ]:
import subprocess, sys
from rulebook.config import settings
from rulebook.store import list_domains

built = set(list_domains(settings.resolved_index_path))
for d in ["checkers", "cornhole"]:
    if d in built:
        print(f"{d}: already built")
    else:
        print(f"{d}: building...")
        subprocess.run([sys.executable, "scripts/build_index.py", "--domain", d], check=True)

## 3. Retrieve

Given a question, embed it the same way and pull the nearest chunks. The **distance** is what the
model actually sorted on (L2 on unit vectors; lower = nearer).

In [ ]:
from rulebook.retrieve import retrieve

for c in retrieve("can one piece capture several times in a single turn?", domain="checkers", k=3):
    print(f"[checkers {c.rule_id}] dist={c.distance:.3f}  {c.text[:72]}")

## 4. Measure it: hit@k and MRR

Retrieval is the half of RAG you can grade **without an LLM** — if you know which rule should
answer each question. That's the gold set in `eval/gold/<domain>.jsonl`. Two metrics:

- **hit@k** — did a correct rule make the top *k*? (Is the answer even in the context window?)
- **MRR** — `1 / (rank of the first correct rule)`. Rewards ranking the answer *high*, not just
  including it.

The same `evaluate()` runs here and in `python -m rulebook.eval`, so the notebook can't drift
from the app.

In [ ]:
from rulebook.eval import load_gold, evaluate

res = evaluate(load_gold("checkers"), k=5)
print(f"checkers   hit@5 = {res.hit_rate:.3f}   MRR@5 = {res.mrr:.3f}   (n={res.n})")
for style, (n, hr, mrr) in res.by_style().items():
    print(f"  {style:<11} n={n}  hit@5={hr:.3f}  MRR={mrr:.3f}")

## 5. Where the local model stumbles

The default local embedder is a **static** model — fast, free, no PyTorch — but it can't read a
word's meaning from context the way a hosted contextual model can. The misses and low ranks below
are that limitation showing up. (Note the `style` split above is only a rough difficulty proxy;
static models don't lean on literal word overlap the way you might expect.)

In [ ]:
misses = [r for r in res.items if not r.hit or r.rr < 1.0]
if not misses:
    print("No misses or low ranks on this run.")
for r in misses:
    flag = "MISS" if not r.hit else f"rank {round(1 / r.rr)}"
    print(f"{flag:>7}  {r.item.question}")
    print(f"         gold={r.item.gold}  top={r.retrieved_ids[:5]}")

## 6. Local vs. a hosted embedder (Voyage)

Here's the payoff: the same gold set turns *"is Voyage worth it?"* into a number. The recipe
(needs a `VOYAGE_API_KEY`, which costs a little):

```bash
# 1. switch provider and rebuild the index with Voyage vectors
EMBEDDING_PROVIDER=voyage EMBEDDING_MODEL=voyage-4 \
    uv run python scripts/build_index.py --domain checkers
# 2. re-run the eval cell above with the same provider set, and compare the rows
```

On a small, clean corpus like these samples the two may land close — **that's the lesson, not a
failure.** You now have *evidence* instead of a guess, and the gap widens as the corpus gets
larger and messier. The cell below just reports which mode you're in; it never spends money on
its own.

In [ ]:
import os

if os.environ.get("VOYAGE_API_KEY") and os.environ.get("EMBEDDING_PROVIDER") == "voyage":
    print("Voyage is active — rebuild the index (recipe above), then re-run the eval cell.")
else:
    print("Local baseline only (no VOYAGE_API_KEY + EMBEDDING_PROVIDER=voyage).")
    print("Follow the recipe above to produce the Voyage row and compare.")

## Takeaway

- Embeddings put **meaning in a geometry**; retrieval is nearest-neighbours in that geometry.
- **hit@k / MRR** are how you *measure* retrieval — and the honest way to decide whether a
  stronger (and costlier) embedder earns its keep.
- Measuring is cheap and key-free; do it before you reach for a bigger model.